# Hypothesis Testing2 - Comprehensive Deployment and Statistical Validation

This notebook executes a full end-to-end pipeline for:
- roadmap-stage execution tracking
- advanced model training (boosting, bagging, stacking)
- statistical hypothesis testing with confidence intervals and p-values
- artifact generation for reporting/deployment


## 1) Environment Setup


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
SCRIPT_DIR = PROJECT_ROOT / 'src' / 'scripts'
if str(SCRIPT_DIR) not in sys.path:
    sys.path.append(str(SCRIPT_DIR))

import hypothesis_testing2 as ht2

print('Project root:', PROJECT_ROOT)
print('Script exists:', (SCRIPT_DIR / 'hypothesis_testing2.py').exists())


## 2) Configure the Run

Tune these parameters for final experiments:
- `n_boot`, `n_perm` for stronger statistical inference
- `max_zones` and `min_zone_train_rows` for zone-level analyses


In [ ]:
args = ht2.build_parser().parse_args([])

# Optional overrides for notebook runs
args.n_boot = 1500
args.n_perm = 2000
args.max_zones = 80
args.min_zone_train_rows = 120
args.alpha = 0.05

print(args)


## 3) Execute Full Hypothesis Testing2 Pipeline


In [ ]:
ht2.run(args)


## 4) Load Generated Artifacts


In [ ]:
OUT_DIR = Path(args.output_dir)
print('Output dir:', OUT_DIR)
print('Files:')
for p in sorted(OUT_DIR.glob('*')):
    print('-', p.name)


In [ ]:
summary_path = OUT_DIR / 'hypothesis_testing2_summary.json'
pairwise_path = OUT_DIR / 'pairwise_model_stat_tests.csv'
model_path = OUT_DIR / 'model_comparison.csv'
roadmap_path = OUT_DIR / 'roadmap_execution_report.json'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
pairwise_df = pd.read_csv(pairwise_path)
model_df = pd.read_csv(model_path)
roadmap = json.loads(roadmap_path.read_text(encoding='utf-8'))

summary


## 5) Roadmap Execution Status


In [ ]:
pd.DataFrame(roadmap)


## 6) Advanced Model Performance Table


In [ ]:
model_df.sort_values('mae')


## 7) Pairwise Statistical Tests (Model Errors)

Interpretation:
- `mean_abs_error_diff_a_minus_b < 0`: model_a has lower absolute error on average
- `p_permutation` and `p_wilcoxon` < alpha indicates statistically meaningful difference


In [ ]:
pairwise_df


## 8) Key Hypothesis Verdicts


In [ ]:
tests = summary.get('statistical_tests', {})

zone_test = tests.get('h2_zone_specific_outperform_citywide', {})
airport_test = tests.get('h2_airport_predictability', {})
borough_test = tests.get('h2_borough_consistency_manhattan_vs_bronx', {})

print('Zone-specific > City-wide:', zone_test)
print('Airport Predictability:', airport_test)
print('Manhattan vs Bronx:', borough_test)


## 9) Visualization Artifact


In [ ]:
from IPython.display import Image, display
plot_path = OUT_DIR / 'hypothesis_testing2_model_metrics.png'
print('Plot exists:', plot_path.exists(), plot_path)
if plot_path.exists():
    display(Image(filename=str(plot_path)))


## 10) Practical Deployment Notes

For production runs, use command line (same pipeline):

```bash
python src/scripts/hypothesis_testing2.py   --input-xlsx data/raw/yellow_taxi_24months_complete.xlsx   --zone-lookup data/external/taxi_zone_lookup.csv   --output-dir reports/results/hypothesis_testing2   --n-boot 3000 --n-perm 5000 --alpha 0.05
```

Recommended production actions:
1. Increase `n_boot` and `n_perm` for stronger confidence in inference.
2. Expand feature set (weather/events/holiday metadata) and rerun.
3. Compare stability across multiple temporal windows.
